Yes. Let’s define the problem cleanly.

## Problem we want to solve

We want to build a **MedRAG-style differential diagnosis system**.

Given a patient case with things like:

* age
* sex
* symptoms
* medical history / antecedents

the system should:

1. **retrieve similar prior cases**
2. **compare likely diseases that look similar**
3. **predict the most likely diagnosis**
4. **list plausible alternatives**
5. **ask the most useful follow-up questions when evidence is incomplete**

That matches the core gap DDXPlus was created for: not just predicting one disease, but handling **differential diagnosis** with symptoms, antecedents, and structured evidence. DDXPlus includes about 1.3 million synthetic patient records with ground-truth pathology and a differential diagnosis, specifically to support this kind of system. ([arXiv][1])

## Why this is the right MedRAG problem

MedRAG is aimed at cases where **multiple diseases have similar manifestations**, so simple nearest-neighbor retrieval or plain QA is not enough. Its goal is to use retrieval plus structured diagnostic knowledge to make the diagnosis more specific and to generate useful follow-up questions when the case is ambiguous. ([arXiv][2])

So the real problem is not:

“Answer a medical yes/no question from one abstract.”

It is:

**“Given an incomplete patient case, distinguish between similar diseases and decide what extra information is needed to reduce uncertainty.”** ([arXiv][1])

## Formal version of the problem

### Input

A patient case:

* demographic details
* symptoms
* antecedents / history
* optionally symptom values that are binary, categorical, or multi-choice

DDXPlus is useful here because it contains exactly this richer symptom structure, not only binary features. ([arXiv][1])

### Output

The system should produce:

* **top diagnosis**
* **top-k differential diagnoses**
* **short explanation**
* **follow-up questions** that would best separate the leading candidates

## Why this problem is hard

The difficulty is:

* different diseases can share many symptoms
* the initial case is often incomplete
* some features matter much more than others
* retrieval may bring back lookalike cases instead of the truly correct disease
* the system must know when to say “not enough evidence yet”

That is why differential diagnosis is a harder and more realistic benchmark than PubMedQA-style yes/no/maybe answering. ([arXiv][1])

## Project statement

A good one-line project statement would be:

**Build a retrieval-augmented differential diagnosis system that, given a patient case, retrieves similar cases and produces a ranked diagnosis list with explanations and follow-up questions.**

## First version we should implement

For version 1, keep it simple:

**Input:** one DDXPlus patient case
**Retriever:** dense search over past patient cases
**Generator:** produce

* most likely diagnosis
* top 3 alternatives
* 3 follow-up questions

That is already a faithful first replica of the MedRAG-style use case. ([arXiv][1])

## Success criteria

We will know it is working if it improves on these:

* top-1 diagnosis accuracy
* top-3 / top-5 diagnosis recall
* quality of differential ranking
* usefulness of follow-up questions

## In one sentence

**The problem is to help with differential diagnosis under uncertainty, not just retrieve a similar text.**

Next, we should define the **exact input and output format** for one DDXPlus case.

[1]: https://arxiv.org/abs/2205.09148?utm_source=chatgpt.com "DDXPlus: A New Dataset For Automatic Medical Diagnosis"
[2]: https://arxiv.org/html/2507.12774v1?utm_source=chatgpt.com "A Systematic Survey of Electronic Health Record Modeling"


In [1]:
!pip install -q datasets pandas

In [6]:


import pandas as pd
from datasets import load_dataset

# load small subset first
ds = load_dataset("aai530-group6/ddxplus", split="train[:20000]")

# convert to dataframe
df = ds.to_pandas()

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

print("\nFirst row:")
display(df.head(1))

Rows: 20000
Columns:
['AGE', 'DIFFERENTIAL_DIAGNOSIS', 'SEX', 'PATHOLOGY', 'EVIDENCES', 'INITIAL_EVIDENCE']

First row:


,AGE,DIFFERENTIAL_DIAGNOSIS,SEX,PATHOLOGY,EVIDENCES,INITIAL_EVIDENCE
0,18,"[['Bronchitis', 0.19171203430383882], ['Pneumo...",M,URTI,"['E_48', 'E_50', 'E_53', 'E_54_@_V_161', 'E_54...",E_91


In [7]:
def make_case_text(row):
    text = ""
    text += "Age: " + str(row["AGE"]) + "\n"
    text += "Sex: " + str(row["SEX"]) + "\n"
    text += "Initial evidence: " + str(row["INITIAL_EVIDENCE"]) + "\n"
    text += "Evidences: " + str(row["EVIDENCES"]) + "\n"
    return text

df["case_text"] = df.apply(make_case_text, axis=1)

print(df["case_text"].iloc[0])

Age: 18
Sex: M
Initial evidence: E_91
Evidences: ['E_48', 'E_50', 'E_53', 'E_54_@_V_161', 'E_54_@_V_183', 'E_55_@_V_89', 'E_55_@_V_108', 'E_55_@_V_167', 'E_56_@_4', 'E_57_@_V_123', 'E_58_@_3', 'E_59_@_3', 'E_77', 'E_79', 'E_91', 'E_97', 'E_201', 'E_204_@_V_10', 'E_222']



# Case Test

In [11]:
import pandas as pd

df_small = df

def make_case_text(row):
    text = ""
    text += "Age: " + str(row["AGE"]) + "\n"
    text += "Sex: " + str(row["SEX"]) + "\n"
    text += "Initial evidence: " + str(row["INITIAL_EVIDENCE"]) + "\n"
    text += "Evidences: " + str(row["EVIDENCES"]) + "\n"
    return text

df_small["case_text"] = df_small.apply(make_case_text, axis=1)
df_small["label"] = df_small["PATHOLOGY"].astype(str)

display(df_small[["case_text", "label"]].head(3))
print(df_small["label"].value_counts().head(10))

,case_text,label
0,Age: 18\nSex: M\nInitial evidence: E_91\nEvide...,URTI
1,Age: 21\nSex: M\nInitial evidence: E_50\nEvide...,HIV (initial infection)
2,Age: 19\nSex: F\nInitial evidence: E_77\nEvide...,Pneumonia


label
URTI                        1206
Viral pharyngitis           1097
Anemia                       949
Allergic sinusitis           783
Localized edema              553
Acute dystonic reactions     546
HIV (initial infection)      511
Guillain-Barré syndrome      510
Pulmonary embolism           508
Anaphylaxis                  505
Name: count, dtype: int64


In [12]:
df_small = df.copy()

train_df = df_small.iloc[:16000].copy()
eval_df = df_small.iloc[16000:20000].copy()

print("Train:", len(train_df))
print("Eval :", len(eval_df))

Train: 16000
Eval : 4000


In [13]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00


In [15]:
from huggingface_hub import login

# Replace "YOUR_HF_TOKEN" with your actual Hugging Face token
login(token="")

In [17]:
#!pip install -q transformers accelerate bitsandbytes sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

#model_name = "OpenMeditron/Meditron3-Qwen2.5-7B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

prompt = """
You are helping with differential diagnosis from patient cases.

Patient case:
Age: 45
Sex: M
Initial evidence: chest pain, shortness of breath
Evidences: pain worse on exertion, sweating, nausea

Give:
1. most likely diagnosis
2. top 3 alternatives
3. 3 follow-up questions

Keep answer short.
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=250
    )

text = tokenizer.decode(output[0], skip_special_tokens=True)
print(text)

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [18]:
!pip install -q -U transformers accelerate sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

messages = [
    {"role": "system", "content": "You are a helpful medical reasoning assistant."},
    {"role": "user", "content": "A 45 year old male has chest pain, sweating, nausea, and shortness of breath. Give likely diagnosis, 3 alternatives, and 3 follow-up questions."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250
    )

generated = outputs[0][inputs.input_ids.shape[1]:]
answer = tokenizer.decode(generated, skip_special_tokens=True)

print(answer)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 1.2 MB/s eta 0:00:00


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

### Likely Diagnosis:
The most probable diagnosis for this patient is **Acute Coronary Syndrome (ACS)**, specifically **Myocardial Infarction (MI)** or **Non-ST Elevation Myocardial Infarction (NSTEMI)**. The symptoms of chest pain, sweating, nausea, and shortness of breath are classic signs of severe cardiac ischemia.

### Alternatives:
1. **Pulmonary Embolism (PE):** This condition can cause similar symptoms, including chest pain, shortness of breath, and sweating. Additionally, patients with PE may experience coughing up blood or hemoptysis.
2. **Aortic Dissection:** This is an acute condition where the inner layer of the aorta tears, causing severe chest pain that can radiate to the back or abdomen. Other symptoms might include dizziness or fainting.
3. **Gastroesophageal Reflux Disease (GERD):** While chest pain can occur due to GERD, it typically does not present with associated symptoms such as sweating, nausea, and shortness of breath. However, these symptoms could be present i

In [20]:
!pip install -q rank-bm25

In [21]:
from rank_bm25 import BM25Okapi
import torch

# build BM25 on train cases
train_texts = train_df["case_text"].astype(str).tolist()
train_labels = train_df["label"].astype(str).tolist()

tokenized_train = []
for text in train_texts:
    tokens = text.lower().split()
    tokenized_train.append(tokens)

bm25 = BM25Okapi(tokenized_train)

# pick one eval case
query_text = str(eval_df.iloc[0]["case_text"])
query_label = str(eval_df.iloc[0]["label"])

# retrieve top 3 similar cases
query_tokens = query_text.lower().split()
scores = bm25.get_scores(query_tokens)
top_indices = scores.argsort()[::-1][:3]

retrieved_text = ""

print("PATIENT CASE")
print("=" * 80)
print(query_text)

print("\nTRUE DIAGNOSIS")
print("=" * 80)
print(query_label)

print("\nTOP 3 RETRIEVED CASES")
print("=" * 80)

for rank, idx in enumerate(top_indices, start=1):
    case_text = train_texts[idx]
    case_label = train_labels[idx]

    print("\nCase", rank)
    print("Diagnosis:", case_label)
    print(case_text[:800])
    print("-" * 80)

    retrieved_text += "Similar case " + str(rank) + "\n"
    retrieved_text += "Diagnosis: " + case_label + "\n"
    retrieved_text += case_text + "\n\n"

# build prompt
messages = [
    {
        "role": "system",
        "content": "You are a helpful medical reasoning assistant."
    },
    {
        "role": "user",
        "content": (
            "Patient case:\n"
            + query_text
            + "\n\nRetrieved similar cases:\n"
            + retrieved_text
            + "\nGive:\n"
            + "1. Most likely diagnosis\n"
            + "2. Top 3 alternative diagnoses\n"
            + "3. Short reasoning\n"
            + "4. 3 follow-up questions\n\n"
            + "Use normal diagnosis names. Keep answer short and clear."
        )
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300
    )

generated = outputs[0][inputs.input_ids.shape[1]:]
answer = tokenizer.decode(generated, skip_special_tokens=True)

print("\nMODEL ANSWER")
print("=" * 80)
print(answer)

PATIENT CASE
Age: 31
Sex: F
Initial evidence: E_201
Evidences: ['E_41', 'E_49', 'E_53', 'E_54_@_V_161', 'E_54_@_V_181', 'E_55_@_V_20', 'E_55_@_V_21', 'E_55_@_V_33', 'E_55_@_V_137', 'E_55_@_V_148', 'E_56_@_4', 'E_57_@_V_123', 'E_58_@_8', 'E_59_@_4', 'E_79', 'E_91', 'E_201', 'E_204_@_V_10', 'E_227']


TRUE DIAGNOSIS
Viral pharyngitis

TOP 3 RETRIEVED CASES

Case 1
Diagnosis: Viral pharyngitis
Age: 65
Sex: F
Initial evidence: E_201
Evidences: ['E_41', 'E_49', 'E_53', 'E_54_@_V_161', 'E_54_@_V_181', 'E_55_@_V_20', 'E_55_@_V_21', 'E_55_@_V_33', 'E_55_@_V_137', 'E_55_@_V_148', 'E_56_@_9', 'E_57_@_V_123', 'E_58_@_8', 'E_59_@_3', 'E_79', 'E_91', 'E_181', 'E_201', 'E_204_@_V_10', 'E_227']

--------------------------------------------------------------------------------

Case 2
Diagnosis: Viral pharyngitis
Age: 65
Sex: F
Initial evidence: E_201
Evidences: ['E_41', 'E_49', 'E_53', 'E_54_@_V_161', 'E_54_@_V_181', 'E_55_@_V_20', 'E_55_@_V_21', 'E_55_@_V_33', 'E_55_@_V_137', 'E_55_@_V_148', 'E_56_@_

#Itertaive RAG Sample

first run on initial patient case

model gives diagnosis + follow-up questions

you type one follow-up answer

case is updated

retrieval runs again

model gives refined diagnosis


In [22]:
import torch

# assume these already exist:
# train_df, bm25, tokenizer, model

train_texts = train_df["case_text"].astype(str).tolist()
train_labels = train_df["label"].astype(str).tolist()

def retrieve_top_cases(case_text, top_k=3):
    query_tokens = case_text.lower().split()
    scores = bm25.get_scores(query_tokens)
    top_indices = scores.argsort()[::-1][:top_k]

    retrieved_text = ""
    retrieved_labels = []

    for rank, idx in enumerate(top_indices, start=1):
        label = train_labels[idx]
        text = train_texts[idx]

        retrieved_labels.append(label)

        retrieved_text += "Similar case " + str(rank) + "\n"
        retrieved_text += "Diagnosis: " + label + "\n"
        retrieved_text += text + "\n\n"

    return retrieved_text, retrieved_labels

def run_rag(case_text):
    retrieved_text, retrieved_labels = retrieve_top_cases(case_text, top_k=3)

    messages = [
        {
            "role": "system",
            "content": "You are a helpful medical reasoning assistant."
        },
        {
            "role": "user",
            "content": (
                "Patient case:\n"
                + case_text
                + "\n\nRetrieved similar cases:\n"
                + retrieved_text
                + "\nGive:\n"
                + "1. Most likely diagnosis\n"
                + "2. Top 3 alternative diagnoses\n"
                + "3. Short reasoning\n"
                + "4. 3 follow-up questions\n\n"
                + "Use normal diagnosis names. Keep answer short and clear."
            )
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300
        )

    generated = outputs[0][inputs.input_ids.shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)

    return answer, retrieved_labels

# -----------------------------
# STEP 1: initial case
# -----------------------------
case_text = str(eval_df.iloc[0]["case_text"])
true_label = str(eval_df.iloc[0]["label"])

print("INITIAL PATIENT CASE")
print("=" * 80)
print(case_text)

print("\nTRUE DIAGNOSIS")
print("=" * 80)
print(true_label)

answer_1, labels_1 = run_rag(case_text)

print("\nROUND 1 ANSWER")
print("=" * 80)
print(answer_1)

print("\nRETRIEVED DIAGNOSES IN ROUND 1")
print("=" * 80)
print(labels_1)

# -----------------------------
# STEP 2: follow-up answer
# -----------------------------
followup_answer = "Patient says chest pain gets worse on exertion."

updated_case_text = case_text + "\nFollow-up information: " + followup_answer

print("\nUPDATED PATIENT CASE")
print("=" * 80)
print(updated_case_text)

answer_2, labels_2 = run_rag(updated_case_text)

print("\nROUND 2 ANSWER")
print("=" * 80)
print(answer_2)

print("\nRETRIEVED DIAGNOSES IN ROUND 2")
print("=" * 80)
print(labels_2)

INITIAL PATIENT CASE
Age: 31
Sex: F
Initial evidence: E_201
Evidences: ['E_41', 'E_49', 'E_53', 'E_54_@_V_161', 'E_54_@_V_181', 'E_55_@_V_20', 'E_55_@_V_21', 'E_55_@_V_33', 'E_55_@_V_137', 'E_55_@_V_148', 'E_56_@_4', 'E_57_@_V_123', 'E_58_@_8', 'E_59_@_4', 'E_79', 'E_91', 'E_201', 'E_204_@_V_10', 'E_227']


TRUE DIAGNOSIS
Viral pharyngitis

ROUND 1 ANSWER
1. Most likely diagnosis: Viral pharyngitis
Top 3 alternative diagnoses: Acute laryngitis, Bacterial pharyngitis, Laryngotracheitis

2. Short reasoning:
The most similar cases have the diagnosis of viral pharyngitis, and the patient is female with no specific signs of acute laryngitis or bacterial infection.

3. Follow-up questions:
- Are there any symptoms of fever or difficulty swallowing?
- Has the patient experienced similar symptoms in the past?
- Is there any recent exposure to colds or flu?

RETRIEVED DIAGNOSES IN ROUND 1
['Viral pharyngitis', 'Viral pharyngitis', 'Acute laryngitis']

UPDATED PATIENT CASE
Age: 31
Sex: F
Initial

In [23]:
!pip install -q gradio rank-bm25

In [24]:
#!pip install -q gradio rank-bm25

import torch
import gradio as gr

# simple data lists
train_texts = train_df["case_text"].astype(str).tolist()
train_labels = train_df["label"].astype(str).tolist()

def retrieve_top_cases(case_text, top_k=3):
    query_tokens = case_text.lower().split()
    scores = bm25.get_scores(query_tokens)
    top_indices = scores.argsort()[::-1][:top_k]

    retrieved_text = ""
    retrieved_view = ""

    for rank, idx in enumerate(top_indices, start=1):
        label = train_labels[idx]
        text = train_texts[idx]

        retrieved_text += "Similar case " + str(rank) + "\n"
        retrieved_text += "Diagnosis: " + label + "\n"
        retrieved_text += text + "\n\n"

        retrieved_view += "Case " + str(rank) + "\n"
        retrieved_view += "Diagnosis: " + label + "\n"
        retrieved_view += text[:600] + "\n"
        retrieved_view += "-" * 60 + "\n"

    return retrieved_text, retrieved_view

def generate_answer(case_text, retrieved_text):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful medical reasoning assistant."
        },
        {
            "role": "user",
            "content": (
                "Patient case:\n"
                + case_text
                + "\n\nRetrieved similar cases:\n"
                + retrieved_text
                + "\nGive:\n"
                + "1. Most likely diagnosis\n"
                + "2. Top 3 alternative diagnoses\n"
                + "3. Short reasoning\n"
                + "4. 3 follow-up questions\n\n"
                + "Use normal diagnosis names. Keep answer short and clear."
            )
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300
        )

    generated = outputs[0][inputs.input_ids.shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)

    return answer

def run_first_step(case_text):
    retrieved_text, retrieved_view = retrieve_top_cases(case_text, top_k=3)
    answer = generate_answer(case_text, retrieved_text)
    return answer, retrieved_view, case_text

def run_followup_step(case_text, followup_text):
    if followup_text.strip() == "":
        return "Please enter follow-up information.", "", case_text

    updated_case_text = case_text + "\nFollow-up information: " + followup_text
    retrieved_text, retrieved_view = retrieve_top_cases(updated_case_text, top_k=3)
    answer = generate_answer(updated_case_text, retrieved_text)
    return answer, retrieved_view, updated_case_text

with gr.Blocks() as demo:
    gr.Markdown("# Simple Medical RAG Demo")
    gr.Markdown("Step 1: enter patient case and run RAG. Step 2: add follow-up information only if needed.")

    case_input = gr.Textbox(
        label="Patient Case",
        lines=10,
        placeholder="Enter age, sex, symptoms, history, and evidence"
    )

    run_button = gr.Button("Run Initial RAG")

    first_answer = gr.Textbox(
        label="Initial Answer",
        lines=14
    )

    retrieved_cases = gr.Textbox(
        label="Retrieved Similar Cases",
        lines=16
    )

    current_case_state = gr.State("")

    gr.Markdown("## Follow-up")
    followup_input = gr.Textbox(
        label="Follow-up Information",
        lines=4,
        placeholder="Enter extra information only if follow-up is needed"
    )

    refine_button = gr.Button("Run Follow-up RAG")

    refined_answer = gr.Textbox(
        label="Updated Answer",
        lines=14
    )

    refined_retrieval = gr.Textbox(
        label="Updated Retrieved Cases",
        lines=16
    )

    run_button.click(
        fn=run_first_step,
        inputs=case_input,
        outputs=[first_answer, retrieved_cases, current_case_state]
    )

    refine_button.click(
        fn=run_followup_step,
        inputs=[current_case_state, followup_input],
        outputs=[refined_answer, refined_retrieval, current_case_state]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9de7fa4cb0ced22009.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
